In [24]:
import numpy as np 
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import itertools
from statsmodels.tsa.seasonal import seasonal_decompose

In [16]:
warnings.filterwarnings('ignore')

In [23]:
import pandas as pd

class DataCleaner:
    def __init__(self, file_path):
        """Initialize with file path and load the dataset."""
        self.file_path = file_path
        self.df = pd.read_csv(self.file_path)

    def clean_timestamps(self):
        """Automatically detect and convert timestamp columns."""
        date_columns = []
        for col in self.df.columns:
            try:
                sample_values = self.df[col].dropna().astype(str).head(10)
                converted_sample = pd.to_datetime(sample_values, errors='coerce')
                if converted_sample.notna().mean() > 0.7:  # If at least 70% can be converted
                    date_columns.append(col)
            except Exception:
                pass  # Skip columns that raise errors

        for col in date_columns:
            self.df[col] = pd.to_datetime(self.df[col], errors='coerce')

    def handle_missing_values(self, col, value):
        """Fill or drop missing values for a given column."""
        if col in self.df.columns:
            self.df[col].fillna(value, inplace=True)

    def standardize_categorical_data(self, columns):
        """Ensure categorical columns are standardized."""
        for col in columns:
            if col in self.df.columns:
                self.df[col] = self.df[col].astype(str).str.strip().str.title()

    def remove_duplicates(self):
        """Remove duplicate entries based on 'Unique ID'."""
        if "Unique ID" in self.df.columns:
            self.df.drop_duplicates(subset=["Unique ID"], keep="first", inplace=True)

    def convert_numeric_columns(self):
        """Convert appropriate columns to numeric types."""
        numeric_object_cols = [
            col for col in self.df.select_dtypes(include=['object']).columns
            if self.df[col].apply(lambda x: pd.to_numeric(x, errors='coerce')).notna().mean() > 0.7
        ]
        for col in numeric_object_cols:
            self.df[col] = pd.to_numeric(self.df[col], errors='coerce')

    def save_cleaned_data(self, output_path):
        """Save the cleaned data to a new CSV file."""
        self.df.to_csv(output_path, index=False)


In [26]:
class EDA:
    def __init__(self, file_path):
        """Initialize with file path and load the dataset."""
        self.file_path = file_path
        self.df = pd.read_csv(self.file_path)

    def show_missing_values(self, title="Dataset"):
        """Illustrate the missing values using a heatmap."""
        fig1 = px.imshow(self.df.isnull(),
                         color_continuous_scale=['gray', 'blue'],
                         labels=dict(color="NaN"),
                         title=f'Missing Values Heatmap for {title}')
        fig1.update_layout(width=700, height=700)
        fig1.update_coloraxes(showscale=True)
        fig1.show()

        missing_values = self.df.isna().sum()
        print(f'\nMissing Values in {title}:\n{missing_values}')

    def display_column_value_counts(self):
        """Display the top 10 most frequent values for each column."""
        for column in self.df.columns:
            print('-' * 30)
            print(f'Column: {column}')
            print(self.df[column].value_counts().head(10))

    def show_data_distribution(self):
        """Display data distributions using histograms, box plots, violin plots, and scatter plots."""
        numeric_columns = self.df.select_dtypes(include=[np.number]).columns.tolist()

        print('*' * 33)
        print('********** Distributions **********')

        # Pie Charts for Categorical Columns
        for column in self.df.columns:
            if column not in numeric_columns:
                counts = self.df[column].value_counts()
                if len(counts) <= 10:  # Only plot if unique values are manageable
                    fig = px.pie(names=counts.index,
                                 values=counts.values,
                                 title=f'Distribution of {column}')
                    fig.update_layout(height=500)
                    fig.show()

        print('*' * 33)
        print('********** Histograms **********')

        # Histograms for Numeric Columns
        for col in numeric_columns:
            fig = px.histogram(self.df, x=col, title=f'Histogram of {col}')
            fig.update_layout(height=500)
            fig.show()

        print('*' * 33)
        print('********** Box Plots **********')

        # Box Plots for Numeric Columns
        for column in numeric_columns:
            fig = px.box(self.df, y=column, title=f'Box Plot of {column}')
            fig.update_layout(height=500, width=500)
            fig.show()

        print('*' * 33)
        print('********** Violin Plots **********')

        # Violin Plots for Numeric Columns
        for column in numeric_columns:
            fig = px.violin(self.df, y=column, title=f"Violin Plot of {column}")
            fig.update_layout(width=500, height=500)
            fig.show()

        print('*' * 33)
        print('********* Scatter Plots *********')

        # Scatter Plots for Numeric Columns
        for col1, col2 in itertools.combinations(numeric_columns, 2):
            fig = px.scatter(self.df, x=col1, y=col2, title=f'Scatter Plot of {col1} vs {col2}')
            fig.update_layout(height=500)
            fig.show()

        print('*' * 33)
        print('********* Scatter Plot With Trend Lines *********')

        # Scatter Plots with Trend Lines
        for col1, col2 in itertools.combinations(numeric_columns, 2):
            fig = px.scatter(self.df,
                             x=col1,
                             y=col2,
                             title=f'Scatter Plot of {col1} Vs {col2} With The Trend Line',
                             trendline='ols')
            fig.update_traces(line=dict(color='red', width=3))
            fig.update_layout(height=500)
            fig.show()

    def show_seasonal_decomposition(self):
        """Perform seasonal decomposition on numeric columns."""
        numeric_columns = self.df.select_dtypes(include=[np.number]).columns.tolist()

        for column in numeric_columns:
            df_copy = self.df[column].copy()

            # Handle missing values by interpolation
            if df_copy.isnull().any():
                df_copy = df_copy.interpolate()

            df_copy = df_copy.dropna()

            # Ensure no infinite values
            if not np.isfinite(df_copy).all():
                print(f'Column {column} contains non-finite values, decomposition is skipped.')
                continue

            # Perform Seasonal Decomposition
            try:
                decomposition = seasonal_decompose(df_copy, model='additive', period=12)
                fig = decomposition.plot()

                plt.gcf().set_size_inches(10, 6)
                plt.suptitle(f'Decomposition of the temporal series of {column}', fontsize=16, y=1.05)
                plt.show()
            except ValueError:
                print(f"Skipping decomposition for {column}, not enough data points.")


In [29]:
file_path ='./data/DonationAnalysis-Chingu_VoyageSchedules_20250305.csv'

In [30]:
data_cleaner = Data_cleaner(file_path= file_path)